# DocLite: Knowledge Distillation for Document Understanding

**Teacher:** LayoutLMv3 → **Student:** LiLT (distilled)

Experiments: baselines + distillation on FUNSD and SROIE

In [1]:
# Setup
!rm -rf doclite
!git clone https://github.com/lukeengel/doclite.git
%cd doclite
!pip install -r requirements.txt -q
!apt-get install -y tesseract-ocr -qq

import torch, gc
print(f"CUDA: {torch.cuda.is_available()} | GPU: {torch.cuda.get_device_name(0)}")
print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Cloning into 'doclite'...
remote: Enumerating objects: 2492, done.
remote: Counting objects: 100% (2492/2492), done.
remote: Compressing objects: 100% (2166/2166), done.
remote: Total 2492 (delta 333), reused 2479 (delta 322), pack-reused 0 (from 0)
Receiving objects: 100% (2492/2492), 17.17 MiB | 24.53 MiB/s, done.
Resolving deltas: 100% (333/333), done.
Updating files: 100% (2832/2832), done.
/content/doclite
CUDA: True | GPU: Tesla T4
GPU Memory: 14.6 GB


In [5]:
# Uncomment if you have SROIE on Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -qo /content/drive/MyDrive/sroie.zip -d data/

In [2]:
import torch, gc
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, LayoutLMv3Config, LayoutLMv3ForTokenClassification
from collections import Counter

from doclite.configs.core import ENV, WEIGHTS
from doclite.data.document_dataset import DocumentDataset
from doclite.data.collate import collate_document_batch
from doclite.eval.evaluate import evaluate_student
from doclite.models.teacher import TeacherModel
from doclite.models.student import StudentModel
from doclite.distill.distill_loss import compute_distill_loss
from build_funsd_examples import load_funsd_split

device = "cuda"
all_results = {}

NUM_EPOCHS = 10
LR = 5e-5

class LayoutLMv3EvalWrapper(torch.nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, input_ids, attention_mask, bbox, **kwargs):
        fwd = dict(input_ids=input_ids, attention_mask=attention_mask, bbox=bbox)
        if 'pixel_values' in kwargs:
            fwd['pixel_values'] = kwargs['pixel_values']
        return {"logits": self.model(**fwd).logits}

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU: {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")

print("Ready")

Ready


---
# FUNSD
---

In [3]:
FUNSD_ROOT = ENV.DATA / "funsd"
tokenizer = AutoTokenizer.from_pretrained("microsoft/layoutlmv3-base")

funsd_train = load_funsd_split(FUNSD_ROOT / "training_data" / "annotations", tokenizer)
funsd_test = load_funsd_split(FUNSD_ROOT / "testing_data" / "annotations", tokenizer)

print(f"Train: {len(funsd_train)} | Test: {len(funsd_test)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Train: 149 | Test: 50


### FUNSD: LayoutLMv3 Baseline (Teacher)

In [7]:
BS = 8
train_loader = DataLoader(DocumentDataset(funsd_train), batch_size=BS, shuffle=True, collate_fn=collate_document_batch)
test_loader = DataLoader(DocumentDataset(funsd_test), batch_size=BS, shuffle=False, collate_fn=collate_document_batch)

config = LayoutLMv3Config.from_pretrained("microsoft/layoutlmv3-base", num_labels=4, output_hidden_states=True, output_attentions=True)
model = LayoutLMv3ForTokenClassification.from_pretrained("microsoft/layoutlmv3-base", config=config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
wrapper = LayoutLMv3EvalWrapper(model).to(device)

all_results["teacher_params"] = sum(p.numel() for p in model.parameters())

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], bbox=batch["bbox"], labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    metrics = evaluate_student(wrapper, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_teacher"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_teacher']['token_f1']:.4f}")

del model, optimizer, wrapper
free_gpu()

model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     | 
-----------------------------------+------------+-
layoutlmv3.embeddings.position_ids | UNEXPECTED | 
classifier.bias                    | MISSING    | 
classifier.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/10 | loss=1.0251 | acc=0.7025 | F1=0.6355
Epoch 2/10 | loss=0.7210 | acc=0.7410 | F1=0.6668
Epoch 3/10 | loss=0.5710 | acc=0.7865 | F1=0.7381
Epoch 4/10 | loss=0.4330 | acc=0.7871 | F1=0.7294
Epoch 5/10 | loss=0.3172 | acc=0.8067 | F1=0.7565
Epoch 6/10 | loss=0.2379 | acc=0.8129 | F1=0.7610
Epoch 7/10 | loss=0.1867 | acc=0.7909 | F1=0.7434
Epoch 8/10 | loss=0.1582 | acc=0.8179 | F1=0.7707
Epoch 9/10 | loss=0.1082 | acc=0.8147 | F1=0.7635
Epoch 10/10 | loss=0.1604 | acc=0.8104 | F1=0.7586

>>> Best F1: 0.7707
GPU: 11.2 GB free / 14.6 GB total


### FUNSD: LiLT Baseline (Student)

In [8]:
model = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=4).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

all_results["student_params"] = sum(p.numel() for p in model.parameters())

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model.model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], bbox=batch["bbox"], labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    metrics = evaluate_student(model, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_student"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_student']['token_f1']:.4f}")

del model, optimizer
free_gpu()

config.json:   0%|          | 0.00/697 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/523M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

LiltForTokenClassification LOAD REPORT from: SCUT-DLVCLab/lilt-roberta-en-base
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight     | UNEXPECTED | 
pooler.dense.bias       | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/10 | loss=0.9875 | acc=0.7069 | F1=0.6459
Epoch 2/10 | loss=0.6725 | acc=0.6838 | F1=0.6141
Epoch 3/10 | loss=0.5552 | acc=0.7438 | F1=0.7024
Epoch 4/10 | loss=0.4217 | acc=0.7780 | F1=0.7228
Epoch 5/10 | loss=0.3069 | acc=0.7822 | F1=0.7210
Epoch 6/10 | loss=0.2510 | acc=0.7773 | F1=0.7266
Epoch 7/10 | loss=0.1911 | acc=0.7806 | F1=0.7239
Epoch 8/10 | loss=0.1530 | acc=0.7817 | F1=0.7241
Epoch 9/10 | loss=0.1612 | acc=0.7913 | F1=0.7371
Epoch 10/10 | loss=0.1229 | acc=0.7685 | F1=0.7146

>>> Best F1: 0.7371
GPU: 9.8 GB free / 14.6 GB total


### FUNSD: DocLite Distillation

Teacher in **float16** (frozen, saves ~half GPU memory) + student in float32.

In [9]:
BS_DISTILL = 4
train_loader = DataLoader(DocumentDataset(funsd_train), batch_size=BS_DISTILL, shuffle=True, collate_fn=collate_document_batch)
test_loader = DataLoader(DocumentDataset(funsd_test), batch_size=BS_DISTILL, shuffle=False, collate_fn=collate_document_batch)

# Teacher in float16 to save memory (it's frozen, no precision needed for gradients)
teacher = TeacherModel("microsoft/layoutlmv3-base", num_labels=4).half().to(device)
for param in teacher.parameters():
    param.requires_grad = False

student = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=4).to(device)
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)

free_gpu()

results = []
for epoch in range(NUM_EPOCHS):
    teacher.eval()
    student.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        model_inputs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"], "bbox": batch["bbox"]}

        with torch.amp.autocast('cuda'):
            teacher_out = teacher(**model_inputs)

        # Cast teacher outputs to float32 for loss computation
        teacher_out = {k: v.float() if torch.is_tensor(v) else tuple(t.float() for t in v) for k, v in teacher_out.items()}

        student_hf_out = student.model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], labels=batch["labels"],
            output_hidden_states=True, output_attentions=True, return_dict=True,
        )
        student_out = {"logits": student_hf_out.logits, "hidden_states": student_hf_out.hidden_states, "attentions": student_hf_out.attentions}

        distill_losses = compute_distill_loss(teacher_out, student_out, attention_mask=batch["attention_mask"])
        loss = distill_losses["loss_total"] + WEIGHTS.DELTA_TASK * student_hf_out.loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if step % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | loss={loss.item():.4f}")

    metrics = evaluate_student(student, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_distill"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_distill']['token_f1']:.4f}")

del teacher, student, optimizer
free_gpu()

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     | 
-----------------------------------+------------+-
layoutlmv3.embeddings.position_ids | UNEXPECTED | 
classifier.bias                    | MISSING    | 
classifier.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

LiltForTokenClassification LOAD REPORT from: SCUT-DLVCLab/lilt-roberta-en-base
Key                     | Status     | 
------------------------+------------+-
embeddings.position_ids | UNEXPECTED | 
pooler.dense.weight     | UNEXPECTED | 
pooler.dense.bias       | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


GPU: 9.7 GB free / 14.6 GB total
  Epoch 1 | Step 0/38 | loss=1.5119
  Epoch 1 | Step 10/38 | loss=1.4279
  Epoch 1 | Step 20/38 | loss=1.1906
  Epoch 1 | Step 30/38 | loss=1.2030
Epoch 1/10 | loss=1.2608 | acc=0.6721 | F1=0.5990
  Epoch 2 | Step 0/38 | loss=1.0007
  Epoch 2 | Step 10/38 | loss=0.8952
  Epoch 2 | Step 20/38 | loss=1.0484
  Epoch 2 | Step 30/38 | loss=1.0988
Epoch 2/10 | loss=1.0339 | acc=0.7413 | F1=0.6856
  Epoch 3 | Step 0/38 | loss=0.7939
  Epoch 3 | Step 10/38 | loss=0.9085
  Epoch 3 | Step 20/38 | loss=0.9746
  Epoch 3 | Step 30/38 | loss=0.7912
Epoch 3/10 | loss=0.9185 | acc=0.7561 | F1=0.7001
  Epoch 4 | Step 0/38 | loss=0.8078
  Epoch 4 | Step 10/38 | loss=0.8443
  Epoch 4 | Step 20/38 | loss=0.9360
  Epoch 4 | Step 30/38 | loss=0.7836
Epoch 4/10 | loss=0.8610 | acc=0.7891 | F1=0.7290
  Epoch 5 | Step 0/38 | loss=0.8397
  Epoch 5 | Step 10/38 | loss=0.6534
  Epoch 5 | Step 20/38 | loss=0.8184
  Epoch 5 | Step 30/38 | loss=0.6812
Epoch 5/10 | loss=0.7831 | acc=0

### FUNSD Results

In [10]:
tp = all_results["teacher_params"]
sp = all_results["student_params"]
ft = all_results["funsd_teacher"]
fs = all_results["funsd_student"]
fd = all_results["funsd_distill"]

print("=" * 75)
print("FUNSD RESULTS")
print("=" * 75)
print(f"{'Model':<30s} {'Acc':>8s} {'F1 macro':>10s} {'F1 micro':>10s} {'Params':>12s}")
print("-" * 75)
print(f"{'LayoutLMv3 (teacher)':<30s} {ft['token_acc']:>8.4f} {ft['token_f1_macro']:>10.4f} {ft['token_f1_micro']:>10.4f} {tp:>12,}")
print(f"{'LiLT (baseline)':<30s} {fs['token_acc']:>8.4f} {fs['token_f1_macro']:>10.4f} {fs['token_f1_micro']:>10.4f} {sp:>12,}")
print(f"{'LiLT + DocLite (ours)':<30s} {fd['token_acc']:>8.4f} {fd['token_f1_macro']:>10.4f} {fd['token_f1_micro']:>10.4f} {sp:>12,}")
print("=" * 75)
gap = (fd['token_f1'] - fs['token_f1']) / (ft['token_f1'] - fs['token_f1'] + 1e-8) * 100
print(f"Distillation closed {gap:.1f}% of the teacher-student F1 gap")
print(f"Compression: {tp/sp:.1f}x fewer parameters")

FUNSD RESULTS
Model                               Acc   F1 macro   F1 micro       Params
---------------------------------------------------------------------------
LayoutLMv3 (teacher)             0.8179     0.7707     0.8179  125,330,052
LiLT (baseline)                  0.7913     0.7371     0.7913  130,167,492
LiLT + DocLite (ours)            0.7891     0.7290     0.7891  130,167,492
Distillation closed -24.1% of the teacher-student F1 gap
Compression: 1.0x fewer parameters


---
# FUNSD: Multimodal Distillation (Teacher sees images)

Teacher sees **text + layout + image** → Student sees only **text + layout**

---

In [18]:
!apt-get install -y tesseract-ocr -qq
!pip install pytesseract -q


In [4]:
TRAIN_IMG_DIR = FUNSD_ROOT / "training_data" / "images"
TEST_IMG_DIR = FUNSD_ROOT / "testing_data" / "images"

print("Loading data with images...")
funsd_train_img = load_funsd_split(FUNSD_ROOT / "training_data" / "annotations", tokenizer, image_dir=TRAIN_IMG_DIR)
funsd_test_img = load_funsd_split(FUNSD_ROOT / "testing_data" / "annotations", tokenizer, image_dir=TEST_IMG_DIR)
print(f"Done. Has pixel_values: {'pixel_values' in funsd_train_img[0]}")

Loading data with images...
Done. Has pixel_values: True


### LayoutLMv3 + Image Baseline

In [5]:
BS = 4  # smaller batch for image data (more memory per sample)
train_loader = DataLoader(DocumentDataset(funsd_train_img), batch_size=BS, shuffle=True, collate_fn=collate_document_batch)
test_loader = DataLoader(DocumentDataset(funsd_test_img), batch_size=BS, shuffle=False, collate_fn=collate_document_batch)

config = LayoutLMv3Config.from_pretrained("microsoft/layoutlmv3-base", num_labels=4, output_hidden_states=True, output_attentions=True)
model = LayoutLMv3ForTokenClassification.from_pretrained("microsoft/layoutlmv3-base", config=config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
wrapper = LayoutLMv3EvalWrapper(model).to(device)

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
                    bbox=batch["bbox"], pixel_values=batch.get("pixel_values"), labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    metrics = evaluate_student(wrapper, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_teacher_img"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_teacher_img']['token_f1']:.4f}")

del model, optimizer, wrapper
free_gpu()

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     | 
-----------------------------------+------------+-
layoutlmv3.embeddings.position_ids | UNEXPECTED | 
classifier.bias                    | MISSING    | 
classifier.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/10 | loss=0.8860 | acc=0.6918 | F1=0.6729
Epoch 2/10 | loss=0.6660 | acc=0.7808 | F1=0.7311
Epoch 3/10 | loss=0.4800 | acc=0.7882 | F1=0.7430
Epoch 4/10 | loss=0.3632 | acc=0.8024 | F1=0.7459
Epoch 5/10 | loss=0.2944 | acc=0.8051 | F1=0.7564
Epoch 6/10 | loss=0.2045 | acc=0.7986 | F1=0.7541
Epoch 7/10 | loss=0.1950 | acc=0.7812 | F1=0.7374
Epoch 8/10 | loss=0.1663 | acc=0.7906 | F1=0.7381
Epoch 9/10 | loss=0.1871 | acc=0.7883 | F1=0.7329
Epoch 10/10 | loss=0.1665 | acc=0.7744 | F1=0.7114

>>> Best F1: 0.7564
GPU: 11.9 GB free / 14.6 GB total


### Multimodal DocLite Distillation

In [6]:
BS_DISTILL = 2
train_loader = DataLoader(DocumentDataset(funsd_train_img), batch_size=BS_DISTILL, shuffle=True, collate_fn=collate_document_batch)
test_loader = DataLoader(DocumentDataset(funsd_test_img), batch_size=BS_DISTILL, shuffle=False, collate_fn=collate_document_batch)

teacher = TeacherModel("microsoft/layoutlmv3-base", num_labels=4).half().to(device)
for param in teacher.parameters():
    param.requires_grad = False

student = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=4).to(device)
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)

free_gpu()

results = []
for epoch in range(NUM_EPOCHS):
    teacher.eval()
    student.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        teacher_inputs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"], "bbox": batch["bbox"]}
        if "pixel_values" in batch:
            teacher_inputs["pixel_values"] = batch["pixel_values"]

        with torch.amp.autocast('cuda'):
            teacher_out = teacher(**teacher_inputs)
        teacher_out = {k: v.float() if torch.is_tensor(v) else tuple(t.float() for t in v) for k, v in teacher_out.items()}

        student_hf_out = student.model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], labels=batch["labels"],
            output_hidden_states=True, output_attentions=True, return_dict=True,
        )
        student_out = {"logits": student_hf_out.logits, "hidden_states": student_hf_out.hidden_states, "attentions": student_hf_out.attentions}

        distill_losses = compute_distill_loss(teacher_out, student_out, attention_mask=batch["attention_mask"])
        loss = distill_losses["loss_total"] + WEIGHTS.DELTA_TASK * student_hf_out.loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if step % 10 == 0:
            print(f"  Epoch {epoch+1} | Step {step}/{len(train_loader)} | loss={loss.item():.4f}")

    metrics = evaluate_student(student, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["funsd_mm_distill"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['funsd_mm_distill']['token_f1']:.4f}")

del teacher, student, optimizer
free_gpu()

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     | 
-----------------------------------+------------+-
layoutlmv3.embeddings.position_ids | UNEXPECTED | 
classifier.bias                    | MISSING    | 
classifier.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

LiltForTokenClassification LOAD REPORT from: SCUT-DLVCLab/lilt-roberta-en-base
Key                     | Status     | 
------------------------+------------+-
pooler.dense.bias       | UNEXPECTED | 
pooler.dense.weight     | UNEXPECTED | 
embeddings.position_ids | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


GPU: 11.7 GB free / 14.6 GB total
  Epoch 1 | Step 0/75 | loss=1.5665
  Epoch 1 | Step 10/75 | loss=1.2649
  Epoch 1 | Step 20/75 | loss=1.2544
  Epoch 1 | Step 30/75 | loss=1.1895
  Epoch 1 | Step 40/75 | loss=1.1347
  Epoch 1 | Step 50/75 | loss=0.8464
  Epoch 1 | Step 60/75 | loss=0.9926
  Epoch 1 | Step 70/75 | loss=1.0711
Epoch 1/10 | loss=1.1687 | acc=0.7146 | F1=0.6474
  Epoch 2 | Step 0/75 | loss=0.9174
  Epoch 2 | Step 10/75 | loss=0.9618
  Epoch 2 | Step 20/75 | loss=0.9406
  Epoch 2 | Step 30/75 | loss=0.7947
  Epoch 2 | Step 40/75 | loss=0.9680
  Epoch 2 | Step 50/75 | loss=0.8380
  Epoch 2 | Step 60/75 | loss=1.1360
  Epoch 2 | Step 70/75 | loss=0.8653
Epoch 2/10 | loss=1.0139 | acc=0.6923 | F1=0.5926
  Epoch 3 | Step 0/75 | loss=0.8211
  Epoch 3 | Step 10/75 | loss=0.7738
  Epoch 3 | Step 20/75 | loss=1.0165
  Epoch 3 | Step 30/75 | loss=1.1319
  Epoch 3 | Step 40/75 | loss=1.0004
  Epoch 3 | Step 50/75 | loss=0.9873
  Epoch 3 | Step 60/75 | loss=0.7605
  Epoch 3 | Step 7

---
# SROIE (Receipts)

Skip if SROIE not uploaded.

---

In [7]:
from build_sroie_examples import load_sroie_split, NUM_LABELS as NUM_LABELS_SROIE

SROIE_ROOT = ENV.DATA / "sroie"
sroie_train = load_sroie_split(SROIE_ROOT/"train"/"box", SROIE_ROOT/"train"/"entities", tokenizer)
sroie_test = load_sroie_split(SROIE_ROOT/"test"/"box", SROIE_ROOT/"test"/"entities", tokenizer)
print(f"SROIE — Train: {len(sroie_train)} | Test: {len(sroie_test)}")

ID2LABEL_SROIE = {0: "company", 1: "date", 2: "address", 3: "total", 4: "other", -100: "[PAD]"}
all_labels_s = [l for ex in sroie_train for l in ex["labels"] if l != -100]
counts_s = Counter(all_labels_s)
print("\nLabel distribution:")
for lid, c in sorted(counts_s.items()):
    print(f"  {ID2LABEL_SROIE[lid]:>10s}: {c:>5d} ({c/len(all_labels_s)*100:.1f}%)")

SROIE — Train: 626 | Test: 347

Label distribution:
     company:    26 (0.1%)
        date:   132 (0.4%)
     address:    81 (0.2%)
       total:  1393 (4.2%)
       other: 31908 (95.1%)


### SROIE: Teacher Baseline

In [ ]:
BS = 8
train_loader = DataLoader(DocumentDataset(sroie_train), batch_size=BS, shuffle=True, collate_fn=collate_document_batch)
test_loader = DataLoader(DocumentDataset(sroie_test), batch_size=BS, shuffle=False, collate_fn=collate_document_batch)

config = LayoutLMv3Config.from_pretrained("microsoft/layoutlmv3-base", num_labels=NUM_LABELS_SROIE, output_hidden_states=True, output_attentions=True)
model = LayoutLMv3ForTokenClassification.from_pretrained("microsoft/layoutlmv3-base", config=config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
wrapper = LayoutLMv3EvalWrapper(model).to(device)

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], bbox=batch["bbox"], labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    metrics = evaluate_student(wrapper, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["sroie_teacher"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['sroie_teacher']['token_f1']:.4f}")

del model, optimizer, wrapper
free_gpu()

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     | 
-----------------------------------+------------+-
layoutlmv3.embeddings.position_ids | UNEXPECTED | 
classifier.bias                    | MISSING    | 
classifier.weight                  | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1/10 | loss=0.1722 | acc=0.9652 | F1=0.3380
Epoch 2/10 | loss=0.0722 | acc=0.9793 | F1=0.6535
Epoch 3/10 | loss=0.0457 | acc=0.9869 | F1=0.6695
Epoch 4/10 | loss=0.0278 | acc=0.9904 | F1=0.7032
Epoch 5/10 | loss=0.0177 | acc=0.9927 | F1=0.7132
Epoch 6/10 | loss=0.0155 | acc=0.9906 | F1=0.7571


### SROIE: Student Baseline

In [ ]:
model = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS_SROIE).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

results = []
for epoch in range(NUM_EPOCHS):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        out = model.model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], bbox=batch["bbox"], labels=batch["labels"])
        out.loss.backward()
        optimizer.step()
        total_loss += out.loss.item()
    metrics = evaluate_student(model, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["sroie_student"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['sroie_student']['token_f1']:.4f}")

del model, optimizer
free_gpu()

### SROIE: DocLite Distillation

In [ ]:
BS_DISTILL = 4
train_loader = DataLoader(DocumentDataset(sroie_train), batch_size=BS_DISTILL, shuffle=True, collate_fn=collate_document_batch)
test_loader = DataLoader(DocumentDataset(sroie_test), batch_size=BS_DISTILL, shuffle=False, collate_fn=collate_document_batch)

teacher = TeacherModel("microsoft/layoutlmv3-base", num_labels=NUM_LABELS_SROIE).half().to(device)
for param in teacher.parameters():
    param.requires_grad = False

student = StudentModel("SCUT-DLVCLab/lilt-roberta-en-base", num_labels=NUM_LABELS_SROIE).to(device)
optimizer = torch.optim.AdamW(student.parameters(), lr=LR, weight_decay=0.01)

free_gpu()

results = []
for epoch in range(NUM_EPOCHS):
    teacher.eval()
    student.train()
    total_loss = 0.0
    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()

        model_inputs = {"input_ids": batch["input_ids"], "attention_mask": batch["attention_mask"], "bbox": batch["bbox"]}

        with torch.amp.autocast('cuda'):
            teacher_out = teacher(**model_inputs)
        teacher_out = {k: v.float() if torch.is_tensor(v) else tuple(t.float() for t in v) for k, v in teacher_out.items()}

        student_hf_out = student.model(
            input_ids=batch["input_ids"], attention_mask=batch["attention_mask"],
            bbox=batch["bbox"], labels=batch["labels"],
            output_hidden_states=True, output_attentions=True, return_dict=True,
        )
        student_out = {"logits": student_hf_out.logits, "hidden_states": student_hf_out.hidden_states, "attentions": student_hf_out.attentions}

        distill_losses = compute_distill_loss(teacher_out, student_out, attention_mask=batch["attention_mask"])
        loss = distill_losses["loss_total"] + WEIGHTS.DELTA_TASK * student_hf_out.loss

        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    metrics = evaluate_student(student, test_loader, device=device)
    results.append(metrics)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | loss={total_loss/len(train_loader):.4f} | acc={metrics['token_acc']:.4f} | F1={metrics['token_f1']:.4f}")

all_results["sroie_distill"] = max(results, key=lambda x: x['token_f1'])
print(f"\n>>> Best F1: {all_results['sroie_distill']['token_f1']:.4f}")

del teacher, student, optimizer
free_gpu()

---
# Final Summary
---

In [ ]:
tp = all_results.get("teacher_params", 0)
sp = all_results.get("student_params", 0)

print("=" * 80)
print("COMPLETE RESULTS")
print("=" * 80)

def print_row(name, r, inputs, params=None):
    p = f"{params:>12,}" if params else f"{'':>12s}"
    micro = r.get('token_f1_micro', r.get('token_acc', 0))
    macro = r.get('token_f1_macro', r.get('token_f1', 0))
    print(f"{name:<35s} {inputs:<8s} {r['token_acc']:>6.4f} {macro:>8.4f} {micro:>8.4f} {p}")

print(f"\n{'FUNSD (Forms)':^80s}")
print(f"{'Model':<35s} {'Input':<8s} {'Acc':>6s} {'F1 mac':>8s} {'F1 mic':>8s} {'Params':>12s}")
print("-" * 80)
print_row("LayoutLMv3 (teacher)", all_results["funsd_teacher"], "T+L", tp)
print_row("LiLT (baseline)", all_results["funsd_student"], "T+L", sp)
print_row("LiLT + DocLite (ours)", all_results["funsd_distill"], "T+L", sp)
if "funsd_teacher_img" in all_results:
    print_row("LayoutLMv3 + Image", all_results["funsd_teacher_img"], "T+L+I", tp)
if "funsd_mm_distill" in all_results:
    print_row("LiLT + DocLite (multimodal)", all_results["funsd_mm_distill"], "T+L", sp)

if "sroie_teacher" in all_results:
    print(f"\n{'SROIE (Receipts)':^80s}")
    print(f"{'Model':<35s} {'Input':<8s} {'Acc':>6s} {'F1 mac':>8s} {'F1 mic':>8s}")
    print("-" * 80)
    print_row("LayoutLMv3 (teacher)", all_results["sroie_teacher"], "T+L")
    print_row("LiLT (baseline)", all_results["sroie_student"], "T+L")
    print_row("LiLT + DocLite (ours)", all_results["sroie_distill"], "T+L")

print("\n" + "=" * 80)
print(f"T=Text, L=Layout, I=Image | Compression: {tp/sp:.1f}x")
print("=" * 80)